1. Loading candidate profiles from a parquet file
2. Converting each row into a clean profile string
3. Embedding profile text

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: False


In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
from dotenv import load_dotenv
from pathlib import Path
from tqdm.auto import tqdm

load_dotenv(Path('../.env'))
EMBED_API_KEY = os.environ.get('EMBED_API_KEY')
if not EMBED_API_KEY:
    raise ValueError('EMBED_API_KEY not set; add it to .env')

DATASET_PATH = '../data/candidates.parquet'
ID_COLUMN = 'id'
MAX_ROWS = 1000                             # set None for full dataset + batch export
EMBED_BATCH_SIZE = 8
EMBED_API_URL = 'https://qwen-embed-runpod.boliu.info/v1/embeddings'
EMBED_MODEL = 'Qwen/Qwen3-Embedding-0.6B'
PROFILES_PATH = Path('../data/candidate_profiles.parquet')
EMBEDDINGS_PATH = Path('../data/candidate_embeddings.npy')

In [10]:
# Use pyarrow directly: pd.read_parquet re-imports pandas extension types and can
# fail in a long-lived notebook kernel with "pandas.period already defined".
# df = pq.read_table(DATASET_PATH).to_pandas()
df = pd.read_parquet("hf://datasets/lang-uk/recruitment-dataset-candidate-profiles-english/data/train-00000-of-00001.parquet")


print(f'Loaded rows: {len(df):,}')
# display(df.head(3))
# print('Columns:', list(df.columns))


Loaded rows: 210,250


In [11]:
# if you want a smaller dataset for testing
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

print(f"Kept rows: {len(df):,}")

Kept rows: 1,000


In [12]:
def _clean(value) -> str:
    if value is None or pd.isna(value):
        return ''
    return ' '.join(str(value).strip().split())


def row_to_profile(row: pd.Series) -> str:
    """Format a candidate row as labeled profile text for embedding."""
    position = _clean(row.get('Position'))
    moreinfo = _clean(row.get('Moreinfo'))
    cv = _clean(row.get('CV'))
    looking_for = _clean(row.get('Looking For'))
    highlights = _clean(row.get('Highlights'))
    english_level = _clean(row.get('English Level'))
    primary_keyword = _clean(row.get('Primary Keyword'))

    if not cv or cv == moreinfo:
        bio, extra_cv = moreinfo, ''
    elif not moreinfo:
        bio, extra_cv = cv, ''
    else:
        bio = cv if len(cv) > len(moreinfo) else moreinfo
        extra_cv = cv if cv != bio else ''

    exp = row.get('Experience Years')
    if pd.notna(exp):
        years = float(exp)
        experience = f'{years:g} year' if years == 1 else f'{years:g} years'
    else:
        experience = ''

    sections = [
        f'Position: {position}' if position else '',
        f'Bio: {bio}' if bio else '',
        f'Looking for: {looking_for}' if looking_for else '',
        f'Highlights: {highlights}' if highlights else '',
        f'English level: {english_level}' if english_level else '',
        f'Experience: {experience}' if experience else '',
        f'Primary keyword: {primary_keyword}' if primary_keyword else '',
        f'CV: {extra_cv}' if extra_cv else '',
    ]
    return '\n'.join(section for section in sections if section)


In [14]:
# 2. Converting each row into a clean profile string
df['profile_text'] = df.apply(row_to_profile, axis=1)
profile_texts = df['profile_text'].tolist()
# display(df[[ID_COLUMN, 'profile_text']].head(3))
# print(df['profile_text'].iloc[0])

In [15]:


headers = {
    'Authorization': f'Bearer {EMBED_API_KEY}',
    'Content-Type': 'application/json',
}

texts = df['profile_text'].tolist()
all_vectors: list[list[float]] = []
batch_starts = range(0, len(texts), EMBED_BATCH_SIZE)

print(f'Embedding {len(texts):,} profiles via {EMBED_API_URL}')
print(f'Model: {EMBED_MODEL}  |  batch size: {EMBED_BATCH_SIZE}')

for start in tqdm(batch_starts, desc='API batches', unit='batch'):
    chunk = texts[start : start + EMBED_BATCH_SIZE]
    response = requests.post(
        EMBED_API_URL,
        headers=headers,
        json={'model': EMBED_MODEL, 'input': chunk},
        timeout=120,
    )
    response.raise_for_status()
    data = sorted(response.json()['data'], key=lambda item: item['index'])
    all_vectors.extend(item['embedding'] for item in data)

candidate_embeddings = np.array(all_vectors, dtype=np.float32)
print(f'Done. Embeddings shape: {candidate_embeddings.shape}')

Embedding 1,000 profiles via https://qwen-embed-runpod.boliu.info/v1/embeddings
Model: Qwen/Qwen3-Embedding-0.6B  |  batch size: 8


API batches:   0%|          | 0/125 [00:00<?, ?batch/s]

Done. Embeddings shape: (1000, 1024)
